In [ ]:
"""
🦠 Disease Risk Assessment - SVM / Gradient Boosting Classifier
Dataset: Tunisian Agricultural Data
Target: disease_risk (0 = Healthy | 1 = At-Risk)
"""

In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

In [3]:
# 1. LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv('dataset_filtré.csv')
print(f"Dataset shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}\n")

Dataset shape: (2182, 13)
Missing values:
annee                      0
Item                       0
surface_cultivee_ha      922
production_tonnes          0
yield_value              911
rendement_tonnes_ha      911
pluviometrie_mm          730
temperature_moyenne_c    730
humidite_pct             730
ph                       126
azote_N                  126
phosphore_P_mg_kg        126
potassium_K_mg_kg        126
dtype: int64



In [5]:
 #2. TARGET ENGINEERING (Disease Risk Label)
# ─────────────────────────────────────────────
df = df.dropna(subset=['rendement_tonnes_ha'])
print(f"Rows after dropping missing target: {len(df)}")

# Yield z-score per crop
crop_mean        = df.groupby('Item')['rendement_tonnes_ha'].transform('mean')
crop_std         = df.groupby('Item')['rendement_tonnes_ha'].transform('std').fillna(0.1)
df['yield_zscore'] = (df['rendement_tonnes_ha'] - crop_mean) / (crop_std + 1e-5)

# Climate thresholds (70th percentile)
hum_q70  = df['humidite_pct'].quantile(0.70)
temp_q70 = df['temperature_moyenne_c'].quantile(0.70)
rain_q70 = df['pluviometrie_mm'].quantile(0.70)

# Disease risk conditions (agronomic rules)
cond_fungal    = (df['humidite_pct'] > hum_q70) & (df['temperature_moyenne_c'] > temp_q70)
cond_root_rot  = (df['humidite_pct'] > hum_q70) & (df['pluviometrie_mm'] > rain_q70)
cond_low_yield = df['yield_zscore'] < -0.5

df['disease_risk'] = (cond_fungal | cond_root_rot | cond_low_yield).astype(int)

print(f"Disease Risk Distribution:\n{df['disease_risk'].value_counts()}")
print(f"  → ✅ Healthy  (0): {(df['disease_risk']==0).sum()} samples")
print(f"  → ⚠️  At-Risk (1): {(df['disease_risk']==1).sum()} samples\n")

Rows after dropping missing target: 1271
Disease Risk Distribution:
disease_risk
0    792
1    479
Name: count, dtype: int64
  → ✅ Healthy  (0): 792 samples
  → ⚠️  At-Risk (1): 479 samples



In [6]:
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────
le = LabelEncoder()
df['crop_encoded'] = le.fit_transform(df['Item'])

# Climate interactions
df['temp_x_hum']   = df['temperature_moyenne_c'] * df['humidite_pct']
df['temp_x_rain']  = df['temperature_moyenne_c'] * df['pluviometrie_mm']
df['rain_per_hum'] = df['pluviometrie_mm'] / (df['humidite_pct'] + 1e-5)
df['log_surface']  = np.log1p(df['surface_cultivee_ha'])

# NPK ratios
df['NP_ratio']  = df['azote_N'] / (df['phosphore_P_mg_kg'] + 1e-5)
df['NK_ratio']  = df['azote_N'] / (df['potassium_K_mg_kg'] + 1e-5)
df['NPK_total'] = df['azote_N'] + df['phosphore_P_mg_kg'] + df['potassium_K_mg_kg']

# Stress indicators
df['ph_deviation'] = abs(df['ph'] - 7.0)
df['hum_stress']   = (df['humidite_pct'] - 65) ** 2

FEATURES = [
    'crop_encoded', 'annee',
    'surface_cultivee_ha', 'log_surface',
    'pluviometrie_mm', 'temperature_moyenne_c', 'humidite_pct',
    'ph', 'ph_deviation',
    'azote_N', 'phosphore_P_mg_kg', 'potassium_K_mg_kg',
    'NP_ratio', 'NK_ratio', 'NPK_total',
    'temp_x_hum', 'temp_x_rain', 'rain_per_hum', 'hum_stress'
]

TARGET = 'disease_risk'

X = df[FEATURES]
y = df[TARGET]

print(f"\nFeatures used: {FEATURES}")
print(f"Target: {TARGET}")
print(f"X shape: {X.shape}, y shape: {y.shape}\n")


Features used: ['crop_encoded', 'annee', 'surface_cultivee_ha', 'log_surface', 'pluviometrie_mm', 'temperature_moyenne_c', 'humidite_pct', 'ph', 'ph_deviation', 'azote_N', 'phosphore_P_mg_kg', 'potassium_K_mg_kg', 'NP_ratio', 'NK_ratio', 'NPK_total', 'temp_x_hum', 'temp_x_rain', 'rain_per_hum', 'hum_stress']
Target: disease_risk
X shape: (1271, 19), y shape: (1271,)



In [7]:
# 4. TRAIN / TEST SPLIT (stratified)
# ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
# 5. MODEL PIPELINES WITH KNN IMPUTATION
# ─────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

svm_pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler',  StandardScaler()),
    ('model',   SVC(probability=True, class_weight='balanced', random_state=42))
])

gb_pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler',  StandardScaler()),
    ('model',   GradientBoostingClassifier(random_state=42))
])

In [9]:
# 6. HYPERPARAMETER TUNING (SVM)
# ─────────────────────────────────────────────
print("🔍 GridSearchCV for SVM...")
param_grid_svm = {
    'model__C':      [0.1, 1, 10, 100],
    'model__kernel': ['rbf', 'poly'],
    'model__gamma':  ['scale', 'auto']
}

grid_svm = GridSearchCV(
    svm_pipeline, param_grid_svm,
    cv=cv, scoring='f1', n_jobs=-1, verbose=1
)
grid_svm.fit(X_train, y_train)

best_svm = grid_svm.best_estimator_
print(f"\n✅ Best SVM params: {grid_svm.best_params_}")
print(f"   CV F1 (train): {grid_svm.best_score_:.4f}")

🔍 GridSearchCV for SVM...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

✅ Best SVM params: {'model__C': 100, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
   CV F1 (train): 0.7182


In [10]:
# 7. HYPERPARAMETER TUNING (Gradient Boosting)
# ─────────────────────────────────────────────
print("\n🔍 GridSearchCV for Gradient Boosting...")
param_grid_gb = {
    'model__n_estimators':  [200, 400],
    'model__max_depth':     [3, 5],
    'model__learning_rate': [0.05, 0.1],
    'model__subsample':     [0.8, 1.0]
}

grid_gb = GridSearchCV(
    gb_pipeline, param_grid_gb,
    cv=cv, scoring='f1', n_jobs=-1, verbose=1
)
grid_gb.fit(X_train, y_train)

best_gb = grid_gb.best_estimator_
print(f"\n✅ Best GB params: {grid_gb.best_params_}")
print(f"   CV F1 (train): {grid_gb.best_score_:.4f}")



🔍 GridSearchCV for Gradient Boosting...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

✅ Best GB params: {'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__n_estimators': 400, 'model__subsample': 1.0}
   CV F1 (train): 0.7288


In [11]:
# 8. STACKING ENSEMBLE (SVM + GB → LogReg)
# ─────────────────────────────────────────────
print("\n🔗 Training Stacking Ensemble...")

imputer     = KNNImputer(n_neighbors=5)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=FEATURES)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),      columns=FEATURES)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)
X_test_sc  = scaler.transform(X_test_imp)

svm_base = SVC(
    **{k.replace('model__', ''): v for k, v in grid_svm.best_params_.items()},
    probability=True, class_weight='balanced', random_state=42
)
gb_base = GradientBoostingClassifier(
    **{k.replace('model__', ''): v for k, v in grid_gb.best_params_.items()},
    random_state=42
)

stacking = StackingClassifier(
    estimators=[('svm', svm_base), ('gb', gb_base)],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000),
    cv=5,
    n_jobs=-1
)
stacking.fit(X_train_sc, y_train)
print("✅ Stacking model trained.")



🔗 Training Stacking Ensemble...
✅ Stacking model trained.


In [12]:
# 9. EVALUATION
# ─────────────────────────────────────────────
def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)
    try:
        auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    except Exception:
        auc = float('nan')

    acc_tr = accuracy_score(y_tr, y_pred_tr)
    acc_te = accuracy_score(y_te, y_pred_te)
    f1_te  = f1_score(y_te, y_pred_te)

    print(f"\n{'─'*45}")
    print(f"  {name}")
    print(f"{'─'*45}")
    print(f"  Accuracy Train : {acc_tr:.4f}")
    print(f"  Accuracy Test  : {acc_te:.4f}  ← KEY METRIC")
    print(f"  F1 Score  Test : {f1_te:.4f}")
    print(f"  ROC-AUC   Test : {auc:.4f}  ← Classification equivalent of R²")

    cm = confusion_matrix(y_te, y_pred_te)
    tn, fp, fn, tp = cm.ravel()
    print(f"  Confusion Matrix:")
    print(f"               Pred 0    Pred 1")
    print(f"  Actual 0  │  {tn:>5}   │  {fp:>5}  │")
    print(f"  Actual 1  │  {fn:>5}   │  {tp:>5}  │")

    rep = classification_report(y_te, y_pred_te, target_names=['Healthy (0)', 'At-Risk (1)'])
    print(f"\n  Classification Report:")
    for line in rep.split('\n'):
        print("    " + line)

    return acc_te, f1_te, auc

print("\n" + "═"*50)
print("       📊  MODEL EVALUATION RESULTS")
print("═"*50)
acc_svm, f1_svm, auc_svm = evaluate("SVM (tuned)",                         best_svm, X_train,    X_test,    y_train, y_test)
acc_gb,  f1_gb,  auc_gb  = evaluate("Gradient Boosting (tuned)",            best_gb,  X_train,    X_test,    y_train, y_test)
acc_stk, f1_stk, auc_stk = evaluate("Stacking Ensemble (SVM+GB→LogReg)",   stacking, X_train_sc, X_test_sc, y_train, y_test)

print("\n" + "═"*50)
best_acc  = max(acc_svm, acc_gb, acc_stk)
best_name = ["SVM", "Gradient Boosting", "Stacking Ensemble"][np.argmax([acc_svm, acc_gb, acc_stk])]
print(f"  🏆 Best Model    : {best_name}")
print(f"  🏆 Best Accuracy : {best_acc:.4f}")
print(f"  🏆 Best ROC-AUC  : {max(auc_svm, auc_gb, auc_stk):.4f}")
print("═"*50)


══════════════════════════════════════════════════
       📊  MODEL EVALUATION RESULTS
══════════════════════════════════════════════════

─────────────────────────────────────────────
  SVM (tuned)
─────────────────────────────────────────────
  Accuracy Train : 0.8583
  Accuracy Test  : 0.7608  ← KEY METRIC
  F1 Score  Test : 0.6935
  ROC-AUC   Test : 0.7971  ← Classification equivalent of R²
  Confusion Matrix:
               Pred 0    Pred 1
  Actual 0  │    125   │     34  │
  Actual 1  │     27   │     69  │

  Classification Report:
                  precision    recall  f1-score   support
    
     Healthy (0)       0.82      0.79      0.80       159
     At-Risk (1)       0.67      0.72      0.69        96
    
        accuracy                           0.76       255
       macro avg       0.75      0.75      0.75       255
    weighted avg       0.76      0.76      0.76       255
    

─────────────────────────────────────────────
  Gradient Boosting (tuned)
────────────────

In [13]:
# 10. FEATURE IMPORTANCE (Gradient Boosting)
# ─────────────────────────────────────────────
print("\n📌 Feature Importances (Gradient Boosting):")
gb_model    = best_gb.named_steps['model']
importances = pd.Series(gb_model.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=False)
for feat, imp in importances.items():
    bar = "█" * int(imp * 100)
    print(f"  {feat:<25} {imp:.4f}  {bar}")


📌 Feature Importances (Gradient Boosting):
  humidite_pct              0.2881  ████████████████████████████
  log_surface               0.1696  ████████████████
  surface_cultivee_ha       0.1542  ███████████████
  annee                     0.1310  █████████████
  crop_encoded              0.0851  ████████
  hum_stress                0.0211  ██
  NK_ratio                  0.0197  █
  temp_x_hum                0.0189  █
  temp_x_rain               0.0184  █
  temperature_moyenne_c     0.0182  █
  azote_N                   0.0128  █
  pluviometrie_mm           0.0125  █
  phosphore_P_mg_kg         0.0109  █
  NP_ratio                  0.0104  █
  NPK_total                 0.0086  
  rain_per_hum              0.0079  
  potassium_K_mg_kg         0.0057  
  ph_deviation              0.0037  
  ph                        0.0031  


In [14]:
# 11. PREDICTION EXAMPLE
# ─────────────────────────────────────────────
print("\n🦠 Example Prediction:")
sample     = X_test.iloc[[0]]
true_label = y_test.iloc[0]
pred_svm   = best_svm.predict(sample)[0]
pred_gb    = best_gb.predict(sample)[0]
prob_gb    = best_gb.predict_proba(sample)[0]

print(f"  True Label  : {'⚠️  At-Risk' if true_label == 1 else '✅ Healthy'} (class {true_label})")
print(f"  SVM Predict : {'⚠️  At-Risk' if pred_svm  == 1 else '✅ Healthy'} (class {pred_svm})")
print(f"  GB  Predict : {'⚠️  At-Risk' if pred_gb   == 1 else '✅ Healthy'} (class {pred_gb})")
print(f"  GB  Proba   : Healthy={prob_gb[0]:.3f} | At-Risk={prob_gb[1]:.3f}")

print("\n✅ Done! Model ready for deployment.")



🦠 Example Prediction:
  True Label  : ✅ Healthy (class 0)
  SVM Predict : ⚠️  At-Risk (class 1)
  GB  Predict : ⚠️  At-Risk (class 1)
  GB  Proba   : Healthy=0.312 | At-Risk=0.688

✅ Done! Model ready for deployment.
